In [ ]:
import os
import sys
import re
from typing import Union
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import dash_ag_grid as dag
from dash import Dash, html

PATH = os.path.dirname(os.path.abspath(""))
DATA_PATH = os.path.join(os.path.dirname(PATH), "data")

sys.path.append(PATH)

import src.dashboard_helpers.dashboard_utils as ut
import src.dashboard_plots.dashboard_plots_comparison as dbp
import src.dashboard_plots.dashboard_plots_summary as dbps

from src.dashboard_plots.dashboard_plots_review import create_heatmap_input, create_listening_heatmap, create_datatable_input, create_datatable

In [ ]:
data = pd.read_csv(DATA_PATH + '/full_data_2026-10-26.csv')
print(data.shape)
data = data.drop_duplicates(subset=['track_id', 'played_at']).reset_index(drop=True)
print(data.shape)
data.head()

In [ ]:
data['played_at'].max()

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp['played_at'] = pd.to_datetime(tmp['played_at']).dt.strftime('%Y-%m-%d %H:%M:%S')
tmp

In [ ]:
pd.to_datetime(tmp['played_at'][0])

## Data Cleaning

- simplify artist_genres column
    - only use first 
    - only use most prevalent
- fill missing values
    - if a song (same song title + same combined_artist_names) occures more than once but one (or more) are lacking audio features
- convert played_at to Datetime 

#### Fill missing feature values

If a song (same song title + same combined_artist_names) occurs more than once but one (or more) are lacking audio features, then fill the missing values accordingly.

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp

In [ ]:
# Convert played_at to datetime
data['played_at'] = pd.to_datetime(data['played_at'],format='ISO8601').dt.strftime('%Y-%m-%d %H:%M:%S')
data = ut.fill_missing_features(data=data)
data = ut.assign_main_genre(data=data)
print(data.shape)
data = ut.assign_simplified_main_genre(data=data)
print(data.shape)

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp

## Data Enrichment

- track_duration_ms to track_duration_sec
- track_duration_ms to track_duration_min_sec (mm:ss)
- seconds_till_next_song
    - sort all data points by played_at (in ascending order)
    - calculate delta between current and next song in seconds
- song_fully_played_flag
    - IF track_duration_sec 0.95 <= seconds_till_next_song THEN 1 ELSE 0
        - we multiply with 0.95 here account for potential inaccuracies of played_at timestamp
        - besides, if the user listened to 90% of the song, we can consider it fully played

In [ ]:
data = ut.convert_track_duration(data=data)
data = ut.add_fully_played_label(data=data, fully_played_threshold=0.95)
data

## Data Analysis

### 1. Data Table (via dash ag grid)

Desired data points in data table:
- track_name
- track_duration_min_sec
- energy
- first_artist_name
- album_name
- album_image_url
- simplified_main_genre
- played_at_date (derived field from 'played_at')
- played_at_time (derived field from 'played_at')
- track_time_played
- track_fully_played

## 1. Data Table

In [ ]:
# NOTE 1: album_image_url should display the album cover, not the url itself
# NOTE 2: track_energy should display a bar chart that visualizes the energy level of the track    
data_reduced = create_datatable_input(data=data)
data_reduced

In [ ]:
data_reduced['Fully Played'].value_counts()

In [ ]:
# Create data table using dash_ag_grid here
datatable = create_datatable(data_reduced)

# Legacy AG Grid theme CSS is not bundled with dash-ag-grid; load it from the CDN
app = Dash(
    __name__,
    assets_folder="../assets",
    external_stylesheets=[dag.themes.BASE, dag.themes.ALPINE],
)
# styles.css clips the page to 100vh, so give the grid its own scrollable container
app.layout = html.Div(datatable, style={"height": "100vh", "overflow": "auto"})
app.run(jupyter_mode="tab", port=8052)

## 2. Heatmap

In [ ]:
normalized = True

heatmap_df = create_heatmap_input(
    data=data,
    date_lower='2025-03-10',
    date_upper='2026-01-09',
    normalized=normalized
)
create_listening_heatmap(heatmap_df, normalized=normalized)

__Next Steps:__
- create heatmap plot via prompt
- refine heatmap plot
- start creating "review" dashboard (using grilling skill)